<a href="https://colab.research.google.com/github/vinhlq2512/research-code-lab/blob/feat/tapta-data-scalability/notebooks/w18_verification_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 W18: Kiểm Chứng Kỹ Thuật, Benchmark TACRED & Tự Động Hóa Multi-Seed
### Đề tài: Continual Relation Extraction with Task-Aware Prompt Adaptation (TAPTA)

Notebook này phục vụ các đầu việc kiểm chứng kỹ thuật của **Tuần W18** theo kế hoạch nghiên cứu chính thức (**Track A — Primary Research**):
1. **Kiểm chứng P4 trên Benchmark thứ hai TACRED Track A** (41 quan hệ, 8 tác vụ, strict 5-shot, $M=0$, seed 2021).
2. **Kiểm chứng Bộ công cụ chạy đa hạt giống (Multi-Seed Runner)** (`run_multiseed.py`) chuẩn bị sẵn sàng cho Tuần W19.
3. **Đo đạc chi phí tham số, bộ nhớ tĩnh và độ trễ suy luận trên GPU** (`profile_model_efficiency.py`).
4. *(Tùy chọn - Track B)*: Thăm dò mở rộng dữ liệu Scalability Study nếu GPU dôi dư.

## 1. Kiểm tra Môi trường GPU (NVIDIA CUDA Preflight Check)

In [1]:
import torch
print("=" * 60)
print("CUDA / GPU PREFLIGHT CHECK")
print("=" * 60)
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:     {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM:     {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"CUDA Version:   {torch.version.cuda}")
else:
    print("⚠️ CẢNH BÁO: Không tìm thấy GPU CUDA! Vui lòng vào Runtime -> Change runtime type -> Chọn T4 GPU.")
print("=" * 60)

CUDA / GPU PREFLIGHT CHECK
CUDA Available: True
GPU Device:     Tesla T4
Total VRAM:     14.56 GB
CUDA Version:   13.0


## 2. Clone / Cập Nhật Repository `research-code-lab`

In [2]:
import os
REPO_URL = "https://github.com/vinhlq2512/research-code-lab.git"
BRANCH = "feat/tapta-data-scalability"
REPO_DIR = "/content/research-code-lab"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

%cd {REPO_DIR}
!pip install -q transformers pyyaml scikit-learn tabulate

Cloning into '/content/research-code-lab'...
remote: Enumerating objects: 3227, done.
remote: Counting objects: 100% (3227/3227), done.
remote: Compressing objects: 100% (2289/2289), done.
remote: Total 3227 (delta 926), reused 3014 (delta 735), pack-reused 0 (from 0)
Receiving objects: 100% (3227/3227), 18.15 MiB | 18.15 MiB/s, done.
Resolving deltas: 100% (926/926), done.
/content/research-code-lab


## 3. Thực Nghiệm 1: Kiểm Chứng P4 Trên Benchmark Thứ Hai TACRED Track A
*(41 quan hệ, 8 tác vụ, strict 5-shot, Rehearsal-Free $M=0$, seed 2021)*

In [3]:
!python experiments/run_p4.py \
    --config configs/p4_hierarchical_prompt_tree_tacred_5shot_seed2021.yaml \
    --device cuda

Continual Relation Extraction: P4_TACRED (p4_hierarchical_tree_tacred)
Description: Proposed Method P4 (TAPTA Hierarchical Tree) on TACRED Track A: Frozen BERT with 2-tier Hierarchical Prompt Tree (L_root=4, L_leaf=4), Sub-Centroid Task Routing, and Decision-Level Soft Prototype Fusion (Zero Oracle Task ID)
Prompt Tree: L_root=4 | L_leaf=4 | L_total=8
Decision Fusion: K=2 | Temp=0.2
Seed:        2021
Results Dir: /content/research-code-lab/runs/p4_hierarchical_tree/tacred_5shot/seed_2021

[1/4] Loaded Protocol v1 Dataset: 8 continual tasks.
[2026-11-24 22:58:43,757] [INFO] HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
[2026-11-24 22:58:44,007] [INFO] HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased "HTTP/1.1 307 Temporary Redirect"
[2026-11-24 22:58:44,243] [INFO] HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased "HTTP/1.1 200 OK"
[2026-11-24 22:58:44,482] [INFO] HTTP Request: HEAD https://huggingface

## 4. Thực Nghiệm 2: Kiểm Thử Bộ Công Cụ Chạy Đa Hạt Giống (Multi-Seed Runner)
*(Chạy 2 hạt giống 2021 & 2022 để kiểm tra pipeline tổng hợp Mean ± Std trước khi sang W19)*

In [4]:
!python experiments/run_multiseed.py \
    --config configs/p4_hierarchical_prompt_tree_fewrel_5shot_seed2021.yaml \
    --seeds 2021 2022 \
    --device cuda

TAPTA P4 MULTI-SEED EXPERIMENT RUNNER: 2 Seeds [2021, 2022]
Base Config: configs/p4_hierarchical_prompt_tree_fewrel_5shot_seed2021.yaml
Execution Device: cuda

>>> Running Seed 2021 (1/2)...
  Executing: /usr/bin/python3 /content/research-code-lab/experiments/run_p4.py --config configs/p4_hierarchical_prompt_tree_fewrel_5shot_seed2021.yaml --seed 2021 --device cuda
Continual Relation Extraction: P4 (p4_hierarchical_tree)
Description: Proposed Method P4 (TAPTA Hierarchical Tree): Frozen BERT with 2-tier Hierarchical Prompt Tree (L_root=4, L_leaf=4), Sub-Centroid Task Routing, and Decision-Level Soft Prototype Fusion (Zero Oracle Task ID)
Prompt Tree: L_root=4 | L_leaf=4 | L_total=8
Decision Fusion: K=2 | Temp=0.2
Seed:        2021
Results Dir: /content/research-code-lab/runs/p4_hierarchical_tree/fewrel_5shot/seed_2021

[1/4] Loaded Protocol v1 Dataset: 8 continual tasks.
[2026-11-24 23:01:54,412] [INFO] HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased "HTTP/1.1 307 

## 5. Thực Nghiệm 3: Phân Tích Độ Phức Tạp, Tham Số & Bộ Nhớ Tĩnh Trên GPU
*(Chứng minh chỉ tốn 0.0028% tham số/task và ~1.03 MB persistent memory)*

In [5]:
!python scripts/profile_model_efficiency.py

PROFILING EFFICIENCY & RESOURCE FOOTPRINT: TAPTA (P4) VS BASELINES
Loading weights: 100% 199/199 [00:00<00:00, 4168.85it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covarian

## 6. (Tùy Chọn - Track B): Thăm Dò Mở Rộng Dữ Liệu (Scalability Study)
*(Chỉ chạy khi GPU còn nhiều quota và đã hoàn tất các mốc Track A ở trên)*

In [8]:
#@title Lựa chọn thí nghiệm Scalability (Track B) { run: "auto" }
run_scalability = False #@param {type:"boolean"}
scale_setting = "ds50 (50-shot)" #@param ["ds10 (10-shot)", "ds50 (50-shot)", "ds100 (100-shot)", "ds_full (~420-shot)"]

scale_map = {
    "ds10 (10-shot)": "configs/scalability/p4_fewrel_ds10_seed2021.yaml",
    "ds50 (50-shot)": "configs/scalability/p4_fewrel_ds50_seed2021.yaml",
    "ds100 (100-shot)": "configs/scalability/p4_fewrel_ds100_seed2021.yaml",
    "ds_full (~420-shot)": "configs/scalability/p4_fewrel_ds_full_seed2021.yaml",
}

if run_scalability:
    cfg = scale_map[scale_setting]
    print(f"🚀 Đang chạy thí nghiệm Track B: {scale_setting}...")
    !python experiments/run_scalability_study.py --config {cfg} --device cuda --oracle_diagnostic
else:
    print("ℹ️ Bỏ qua Track B để tập trung tối đa tài nguyên cho Track A theo đúng kế hoạch W18.")

ℹ️ Bỏ qua Track B để tập trung tối đa tài nguyên cho Track A theo đúng kế hoạch W18.


## 7. Tổng Hợp Kết Quả & Đóng Gói Tải Về (Download Results Artifact)

In [9]:
import json
import shutil
from pathlib import Path
from google.colab import files

print("=" * 70)
print("TỔNG HỢP KẾT QUẢ KIỂM CHỨNG TUẦN W18")
print("=" * 70)

# Kiểm tra kết quả TACRED
tacred_sum = Path("runs/p4_hierarchical_tree/tacred_5shot/seed_2021/summary.json")
if tacred_sum.exists():
    with tacred_sum.open() as f:
        s = json.load(f)
    print(f"✅ P4 trên TACRED Track A:")
    print(f"   - Final Decision Fusion Accuracy: {s.get('final_decision_fusion_accuracy', 0)*100:.2f}%")
    print(f"   - Average Incremental Acc (AIA): {s.get('average_incremental_accuracy', 0)*100:.2f}%")
    print(f"   - Whole Forgetting:              {s.get('whole_forgetting', 0)*100:.2f}%")
    print(f"   - Router Top-1 Accuracy:         {s.get('final_router_top1_accuracy', 0)*100:.2f}%")
else:
    print("ℹ️ Chưa có kết quả TACRED. (Hãy chạy Mục 3)")

# Kiểm tra kết quả Multi-Seed
ms_sum = Path("runs/p4_hierarchical_tree/fewrel_5shot/multiseed_summary.json")
if ms_sum.exists():
    with ms_sum.open() as f:
        m = json.load(f)
    print(f"Multi-Seed Aggregation ({m.get('num_seeds')} seeds):")
    agg = m.get("aggregated_metrics", {})
    for k, v in agg.items():
        print(f"   - {k}: {v.get('formatted')}")
else:
    print("ℹ️ Chưa có kết quả Multi-Seed. (Hãy chạy Mục 4)")

# Đóng gói toàn bộ thư mục runs/
archive_dir = "runs"
zip_name = "/content/w18_verification_results"
if os.path.exists(archive_dir):
    shutil.make_archive(zip_name, "zip", archive_dir)
    print(f"Đã đóng gói thành công: {zip_name}.zip")
    files.download(f"{zip_name}.zip")
else:
    print("Thư mục runs không tồn tại để đóng gói!")

TỔNG HỢP KẾT QUẢ KIỂM CHỨNG TUẦN W18
✅ P4 trên TACRED Track A:
   - Final Decision Fusion Accuracy: 42.45%
   - Average Incremental Acc (AIA): 63.46%
   - Whole Forgetting:              0.00%
   - Router Top-1 Accuracy:         49.87%
ℹ️ Chưa có kết quả Multi-Seed. (Hãy chạy Mục 4)
Đã đóng gói thành công: /content/w18_verification_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>